In [ ]:
import warnings

import numpy as np
from ipynb.fs.defs.ARMA_GARCH import negloglik_armagarch
from ipynb.fs.defs.GARCH_Facts import GARCH
from IPython.display import clear_output
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')

# Monte Carlo Simulations
- Simulate GARCH Process Data with Fixed Alpha and Beta
- Estimate Alpha and Beta using Negative Log-Likelihood
- Compute Mean Square Errors from the True Values

In [ ]:
def monte_carlo(n_values, alpha1, beta1, n_simulations=500):
    '''Estimate Alpha and Beta for GARCH Simulations'''
    
    results = np.zeros([len(n_values), 4])

    for n in range(len(n_values)):
        a1 = []
        b1 = []

        for i in range(n_simulations):

            # Generate data
            X = GARCH(0.01, 0.01, 0.0001, alpha1, beta1, n_values[n])

            # MLE Optimisation
            init = [0.1 * np.var(X*1000), alpha1, beta1]
            bounds = [(10**-8, None), (0.0, 1.0), (0.0, 1.0)]

            res = minimize(
                negloglik_armagarch,
                x0=init,
                args=(X*1000, 0, 0),
                bounds=bounds,
                tol=10**-10
            )
            
            [alpha_0, alpha_1, beta_1] = res.x
            
            a1.append(alpha_1)
            b1.append(beta_1)
            clear_output()
            print(f'n = {n_values[n]}, Iteration {i}: \n Alpha_1: {alpha_1}, Beta_1: {beta_1}')

        results[n] = mean_squared_error(a1, np.repeat(alpha_1, len(a1))), np.var(a1), mean_squared_error(b1, np.repeat(beta_1, len(b1))), np.var(b1)

    return results

In [6]:
def confint(mu, sigma, confidence=0.95):
    '''Create Confidence Intervals'''
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha/2)
    margin = z * (sigma)
    
    lower = mu - margin
    upper = mu + margin
    
    return lower, upper